# Pairs trading with transaction costs

What is pairs trading? 

[Pairs trading](https://www.investopedia.com/terms/p/pairstrade.asp) is a market neutral trading strategy—it seeks to avoid some form of market risk entirely, typically by hedging—that involves taking simultaneously a long position with a short one in two highly correlated stocks or assets, this is crucial for the strategy, but finding and maintaining two assets that have a high correlation can be challenging.

> The strategy monitors performance of two historically correlated securities. When the correlation between the two securities temporarily weakens, i.e. one stock moves up while the other moves down, the pairs trade would be to short the outperforming stock and to long the underperforming one, betting that the "spread" between the two would eventually converge. [1]

The divergence within a pair can be caused by temporary supply/demand changes, large buy/sell orders for one security, reaction for important news about one of the companies, and so on.

> The reason for the deviated stock to come back to original value is itself an assumption. It is assumed that the pair will have similar business performance as in the past during the holding period of the stock.

**Potential drawbacks of the strategy:**

- The divergence between the prices of the securities can be a rational response to news related to one of the securities, invalidating the reversion to the mean assumption.
- Scarcity of opportunities
- Compete with HFT funds

Depending on the market sector, there exists multiple reasons for which two historically correlated securities can have differences on it's stock prices, e.g., company A's management could be extremely efficient while company B's management could be not efficient, this can cause stock A to rise and stock B to fall, and in this case it is not certain that stock B could go back to its mean since it's a management efficiency issue, hence shorting A and buying B would result in a potential loss. 

**Characteristics** 

- Pairs trading is a mean reverting strategy, betting that the prices will eventually return to historical trends, i.e., overperforming stock will do down to its historical mean and underperforming stock will go up to its historical mean

- Pairs trade can hedge sector and market risks, e.g., in case of a market crash, and the two positions going down with it, the trade would result in a gain on the short position and a loss on the long position, leaving a net profit close to breakeven.

**Potentially correlated pairs:**

- Coca Cola and Pepsi
- Walmart and Target
- Dell and HP
- Ford and General Motors

According to Skiena, highly-correlated pairs often (but not always) come from the same sector because they face similar systematic risks.

**Example:** 

> Pepsi (PEP) and Coca-Cola (KO) are different companies that create a similar product, soda pop. Historically, the two companies have shared similar dips and highs, depending on the soda pop market. If the price of Coca-Cola were to go up a significant amount while Pepsi stayed the same, a pairs trader would buy Pepsi stock and sell Coca-Cola stock, assuming that the two companies would later return to their historical balance point. If the price of Pepsi rose to close that gap in price, the trader would make money on the Pepsi stock, while if the price of Coca-Cola fell, they would make money on having shorted the Coca-Cola stock. [2]

## References
<a id="1">[1]</a> 
Skiena, S. 
[Lecture on pairs trading](https://www3.cs.stonybrook.edu/~skiena/691/lectures/lecture23.pdf)

<a id="2">[2]</a> 
Wikipedia. [Pair trading](https://en.wikipedia.org/wiki/Pairs_trade)




In [2]:
import torch
import torch.nn as nn
import numpy as np
from torch.autograd import grad
import torch.optim as optim
import math

torch.set_default_dtype(torch.float64)

# Model parameters
param = {
    # Initial condition
    'p0': 1,
    "mu": 0.2,
    "sigma": 0.4,
    "theta": 0.1,
    "kappa": 1.0,
    "nu": 0.15,
    "rho": 0.5,
    "r": 0.01,
    "gamma": 5.0,
    # Transaction cost params
    "ap": 1.0005,  # ap = 1 + zeta_p
    "bp": 0.9995,  # bp = 1 - eta_p
    "aq": 1.0005,  # aq = 1 + zeta_q
    "bq": 0.9995,  # bq = 1 - eta_q
    # Domain
    "T": 1.0
}

# A_minus and A_plus
def A_plus(p, x, p_dict=param):
    # A+ = (b_p - a_q * exp(x)) * p
    return (p_dict["bq"] - p_dict["aq"] * torch.exp(x)) * p

def A_minus(p, x, p_dict=param):
    # A- = (a_p - b_q * exp(x)) * p
    return (p_dict["ap"] - p_dict["bq"] * torch.exp(x)) * p

# Terminal payoff, i.e., liquidated value of the portfolio
# J(p(T), x(T), y(T)) = A+ * y 1{y>=0} + A- * y 1{y<0}
def terminal_J(p, x, y):
    A_p = (param["bp"] - param["aq"]*torch.exp(x)) * p  # A+
    A_m = (param["ap"] - param["bq"]*torch.exp(x)) * p  # A-
    return torch.where(y >= 0, A_p * y, A_m * y)

# Neural network for H(t, p, x, y)
class H_NN(nn.Module):
    def __init__(self):
        super(H_NN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(4, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

    def forward(self, t, p, x, y):
        
        # Handle scalars (0D) or 1D inputs
        if t.ndim == 0:
            t, p, x, y = [var.unsqueeze(0).unsqueeze(1) for var in (t, p, x, y)]
        elif t.ndim == 1:
            t, p, x, y = [var.unsqueeze(1) for var in (t, p, x, y)]
        input = torch.cat([t, p, x, y], dim=1)
        return torch.exp(self.net(input)) # Ensure H > 0
        
# Neural networks for boundaries Y_b and Y_s (t, p, x)
class Boundary_NN(nn.Module):
    def __init__(self):
        super(Boundary_NN, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 32),
            nn.Tanh(),
            nn.Linear(32, 32),
            nn.Tanh(),
            nn.Linear(32, 1)
        )

    def forward(self, t, p, x):
        input = torch.cat([t, p, x], dim=1)
        return self.net(input)
    
# Operator L2o (PDE in no-transaction region)
def L2o(H, t, p, x, y, p_dict=param):
    # First derivatives
    H_t = torch.autograd.grad(H, t, grad_outputs=torch.ones_like(H), create_graph=True, allow_unused=True, materialize_grads=True)[0]
    H_p = torch.autograd.grad(H, p, grad_outputs=torch.ones_like(H), create_graph=True, allow_unused=True, materialize_grads=True)[0]
    H_x = torch.autograd.grad(H, x, grad_outputs=torch.ones_like(H), create_graph=True, allow_unused=True, materialize_grads=True)[0]

    # Second derivatives
    H_pp = torch.autograd.grad(H_p, p, grad_outputs=torch.ones_like(H_p), create_graph=True, allow_unused=True, materialize_grads=True)[0]
    H_xx = torch.autograd.grad(H_x, x, grad_outputs=torch.ones_like(H_x), create_graph=True, allow_unused=True, materialize_grads=True)[0]
    H_px = torch.autograd.grad(H_p, x, grad_outputs=torch.ones_like(H_p), create_graph=True, allow_unused=True, materialize_grads=True)[0]

    # Auxiliary variables
    mu_x = p_dict['kappa'] * (p_dict['theta'] - x)
    mu_p = p_dict['mu'] * p
    sigma_xx = p_dict['nu'] ** 2
    sigma_xp = p_dict['rho'] * p_dict['nu'] * p_dict['sigma'] * p
    sigma_pp = (p_dict['sigma'] * p) ** 2
 
    # HJB operator LH

    LH = (
        H_t                          # H_t
        + mu_x * H_x                 # k(theta - x) * H_x
        + mu_p * H_p                 # mu_p * H_p
        + 0.5 * sigma_xx * H_xx      # 0.5 * sigma_xx * H_xx
        + sigma_xp * H_px            # sigma_xp * H_px
        + 0.5 * sigma_pp * H_pp      # 0.5 * sigma_pp * H_pp
    )

    return LH

# Operators L2b and L2s
def L2b(H, t, p, x, y, p_dict=param):
    H_y = torch.autograd.grad(H, y, grad_outputs=torch.ones_like(H), create_graph=True, allow_unused=True, materialize_grads=True)[0]

    # Auxiliary variable
    gamma_exp = p_dict['gamma'] * torch.exp(p_dict['r'] * (p_dict['T'] - t))

    L2bH = H_y + gamma_exp * A_minus(p, x) * H
    return L2bH

def L2s(H, t, p, x, y, p_dict=param):
    H_y = torch.autograd.grad(H, y, grad_outputs=torch.ones_like(H), create_graph=True, allow_unused=True, materialize_grads=True)[0]

    # Auxiliary variable
    gamma_exp = p_dict['gamma'] * torch.exp(p_dict['r'] * (p_dict['T'] - t))

    L2sH = H_y + gamma_exp * A_plus(p, x) * H
    return L2sH

# Explicit H in buy region (y <= Yb)
def H_buy(H_b, t, p, x, y, Yb, p_dict=param):
    # Auxiliar variable
    exp_r = torch.exp(p_dict['r'] * (p_dict['T'] - t))

    return torch.exp(-p_dict['gamma'] * A_minus(p, x) * (y - Yb)) * exp_r * H_b

# Explicit H in sell region (y >= Ys)
def H_sell(H_s, t, p, x, y, Ys, p_dict=param):
    # Auxiliar variable
    exp_r = torch.exp(p_dict['r'] * (p_dict['T'] - t))

    return torch.exp(-p_dict['gamma'] * A_plus(p, x) * (y - Ys)) * exp_r * H_s

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Training setup
h_nn = H_NN().to(device)
yb_nn = Boundary_NN().to(device)
ys_nn = Boundary_NN().to(device)
params = list(h_nn.parameters()) + list(yb_nn.parameters()) + list(ys_nn.parameters())
optimizer = optim.Adam(params, lr=0.001)

# Domain bounds
t_min, t_max = 0.0, param['T']
p_min, p_max = 0.3, 4.0
x_min, x_max = -0.5, 0.5
y_min, y_max = -20.0, 20.0

def sample_points(batch_size):
    # Create tensors of random points within the desired domain bounds
    t = torch.rand(batch_size, 1, device=device) * (t_max - t_min) + t_min
    p = torch.rand(batch_size, 1, device=device) * (p_max - p_min) + p_min
    x = torch.rand(batch_size, 1, device=device) * (x_max - x_min) + x_min
    y = torch.rand(batch_size, 1, device=device) * (y_max - y_min) + y_min
    t.requires_grad = True
    p.requires_grad = True
    x.requires_grad = True
    y.requires_grad = True
    return t, p, x, y

def loss_fn():
    # Generate some random points
    t, p, x, y = sample_points(10000)
    H = h_nn(t, p, x, y)
    Yb = yb_nn(t, p, x)
    Ys = ys_nn(t, p, x)

    # Interior loss (no-transaction region).
    mask_nt = (y >= Yb) & (y <= Ys)

    if mask_nt.any():
      loss_nt = torch.mean(L2o(H[mask_nt], t[mask_nt], p[mask_nt], x[mask_nt], y[mask_nt]) ** 2)
    else:
      loss_nt = torch.tensor(0.0, device=device)

    # Buy region loss
    mask_buy = y < Yb
    H_b = h_nn(t[mask_buy], p[mask_buy], x[mask_buy], Yb[mask_buy])
    loss_buy = torch.mean((H[mask_buy] - H_buy(H_b, t[mask_buy], p[mask_buy], x[mask_buy], y[mask_buy], Yb[mask_buy])) ** 2)

    # Sell region loss
    mask_sell = y > Ys
    H_s = h_nn(t[mask_sell], p[mask_sell], x[mask_sell], Ys[mask_sell])
    loss_sell = torch.mean((H[mask_sell] - H_sell(H_s, t[mask_sell], p[mask_sell], x[mask_sell], y[mask_sell], Ys[mask_sell])) ** 2)

    # Boundary conditions loss
    t_b, p_b, x_b, _ = sample_points(2000)  # Sample for boundaries
    y_b = yb_nn(t_b, p_b, x_b)
    H_yb = h_nn(t_b, p_b, x_b, y_b)
    loss_b = torch.mean(L2b(H_yb, t_b, p_b, x_b, y_b) ** 2)
    y_s = ys_nn(t_b, p_b, x_b)  # Reuse samples
    H_ys = h_nn(t_b, p_b, x_b, y_s)
    loss_s = torch.mean(L2s(H_ys, t_b, p_b, x_b, y_s) ** 2)

    # Terminal condition loss
    t_term = torch.ones(5000, 1, device=device) * param['T']
    p_term = torch.rand(5000, 1, device=device) * (p_max - p_min) + p_min
    x_term = torch.rand(5000, 1, device=device) * (x_max - x_min) + x_min
    y_term = torch.rand(5000, 1, device=device) * (y_max - y_min) + y_min
    H_term = h_nn(t_term, p_term, x_term, y_term)
    loss_term = torch.mean((H_term - torch.exp(-param['gamma'] * terminal_J(p_term, x_term, y_term))) ** 2)

    # Soft constraint for Yb < Ys
    loss_order = torch.mean(torch.relu(Yb - Ys) ** 2)

    return loss_nt + loss_buy + loss_sell + loss_b + loss_s + loss_term + 0.1 * loss_order

# Training loop
for epoch in range(10000):
    optimizer.zero_grad()
    loss = loss_fn()
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

Epoch 0, Loss: 6.556309061872901e+199


KeyboardInterrupt: 